In [1]:
# %% 1. Imports
import sys                                               # path bootstrap
import dataclasses                                       # dataclasses.replace for raw updates
from pathlib import Path                                 # repo root discovery

for _candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (_candidate / "gbp").is_dir() and (_candidate / "tests").is_dir():
        sys.path.insert(0, str(_candidate))
        break

# %% 1. Imports
from gbp.loaders import DataLoaderGraph, DataLoaderMockMinimal
from gbp.build.pipeline import build_model
from gbp.consumers.simulator import (
    OrganicFlowPhase, DemandPhase, Environment, EnvironmentConfig,
)

# %% 2. User config — единственная ручка
mock_config = {"n_stations": 8, "n_trips": 200, "n_days": 3, "num_resources":3, "resource_capacity":100}

# %% 3. Source → Raw → Resolved
import pandas as pd

from gbp.build.pipeline import build_model
from gbp.loaders import DataLoaderGraph, DataLoaderMock, GraphLoaderConfig

mock_config = {
    "n_stations": 10,
    "n_depots": 2,
    "n_timestamps": 168,  # 7 days x 24 hours
    "time_freq": "h",
    "start_date": "2025-01-01",
    "num_resources": 3,
    "resource_capacity": 100,
    "ebike_fraction": 0.3,
    "depot_capacity": 200,
    "seed": 42,
}

mock = DataLoaderMock(mock_config)
loader = DataLoaderGraph(mock, GraphLoaderConfig())
raw = loader.load()
resolved = build_model(raw)

# %% 4. Simulation
simulation_log = Environment(
    resolved,
    EnvironmentConfig(phases=[OrganicFlowPhase()], seed=42, scenario_id="run"),
).run()
log_tables = simulation_log.to_dataframes()

# %% 5. Flow parity check — simulation_flow_log vs resolved.observed_flow
flow_keys = ["period_id", "source_id", "target_id", "commodity_category"]

simulated_flow = (
    log_tables["simulation_flow_log"]
    .query("phase_name == 'ORGANIC_FLOW'")
    .groupby(flow_keys, as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "simulated_quantity"})
)
observed_flow = (
    resolved.observed_flow
    .groupby(flow_keys, as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "observed_quantity"})
)
flow_compare = simulated_flow.merge(observed_flow, on=flow_keys, how="outer")
flow_compare["simulated_quantity"] = flow_compare["simulated_quantity"].fillna(0.0)
flow_compare["observed_quantity"] = flow_compare["observed_quantity"].fillna(0.0)
flow_compare["diff"] = flow_compare["simulated_quantity"] - flow_compare["observed_quantity"]
flow_mismatches = flow_compare.loc[flow_compare["diff"].abs() > 1e-9].reset_index(drop=True)

# %% 6. Inventory parity check — simulation_inventory_log vs resolved.observed_inventory
inventory_keys = ["period_id", "facility_id", "commodity_category"]
observed_facility_ids = resolved.observed_inventory["facility_id"].unique()

simulated_inventory = (
    log_tables["simulation_inventory_log"]
    .loc[lambda d: d["facility_id"].isin(observed_facility_ids)]
    .groupby(inventory_keys, as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "simulated_quantity"})
)
observed_inventory = (
    resolved.observed_inventory
    .groupby(inventory_keys, as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "observed_quantity"})
)
inventory_compare = simulated_inventory.merge(observed_inventory, on=inventory_keys, how="outer")
inventory_compare["simulated_quantity"] = inventory_compare["simulated_quantity"].fillna(0.0)
inventory_compare["observed_quantity"] = inventory_compare["observed_quantity"].fillna(0.0)
inventory_compare["diff"] = inventory_compare["simulated_quantity"] - inventory_compare["observed_quantity"]
inventory_mismatches = inventory_compare.loc[inventory_compare["diff"].abs() > 1e-9].reset_index(drop=True)


2026-04-27 21:49:09 [info     ] load_start                     loader=graph_core
2026-04-27 21:49:10 [debug    ] source_validated               loader=graph_core
2026-04-27 21:49:10 [debug    ] observed_flow_built            loader=graph_core rows=727
2026-04-27 21:49:10 [debug    ] observed_inventory_built       loader=graph_core rows=140
2026-04-27 21:49:10 [info     ] load_done                      facilities=12 loader=graph_core


In [2]:
flow_mismatches

,period_id,source_id,target_id,commodity_category,simulated_quantity,observed_quantity,diff


In [3]:
inventory_mismatches

,period_id,facility_id,commodity_category,simulated_quantity,observed_quantity,diff
